[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 03](README.md)

# OpenMP: tareas, dependencias y granularidad

**Tema:** 03 · **Sesiones:** 14, 15 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cuándo un DAG de tareas expone paralelismo suficiente para compensar el costo de creación y sincronización?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Las tareas expresan un DAG dinámico. Su utilidad depende de que las dependencias sean correctas y el trabajo útil supere el costo de planificación.

**Prerrequisitos.**

- Memoria compartida, carreras y sincronización.
- Compilación C/C++ con advertencias habilitadas.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Representar tareas y dependencias.
- Distinguir task, taskgroup y taskwait.
- Elegir un corte de granularidad mediante medición.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Las tareas expresan trabajo potencialmente diferido y ejecutado por cualquier hilo del equipo.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Las dependencias se asocian a regiones de almacenamiento y forman un DAG.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Una recursión fina puede producir más overhead que trabajo; un cutoff conserva trabajo secuencial en hojas pequeñas.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- fork–join — creación y reunión de trabajo paralelo
- entorno de datos — clasificación shared/private/firstprivate
- granularidad — cantidad de trabajo útil por unidad planificada


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Dag Camino Critico

![DAG con trabajo y camino crítico](../../images/dag-camino-critico.svg)

**Cómo leerlo.** El trazo destacado une las tareas que determinan el span. La rama B puede terminar antes sin reducir el total mientras A siga siendo más larga.

### Metodo Rendimiento

![Ciclo de medición, resumen, perfil e hipótesis](../../images/metodo-rendimiento.svg)

**Cómo leerlo.** Una medición se repite y resume antes de perfilar. La conclusión genera un experimento nuevo cambiando una sola variable controlada.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "03"
NOTEBOOK = "03_openmp/03_tareas_rendimiento.ipynb"
assert (ROOT / "curso" / "notebooks" / "03_openmp" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Camino crítico de tareas

**Situación.** Se calcula el tiempo mínimo ideal de un DAG de bloques.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
duration = {"A": 3, "B": 5, "C": 2, "D": 4, "E": 1}
deps = {"A": [], "B": [], "C": ["A"], "D": ["A", "B"], "E": ["C", "D"]}
finish = {}
for task in duration:
    finish[task] = duration[task] + max((finish[p] for p in deps[task]), default=0)
work, span = sum(duration.values()), max(finish.values())
assert (work, span) == (15, 10)
print({"work": work, "span": span, "ideal_parallelism": work/span})


### Explicación del resultado

El span revela si más hilos pueden ayudar antes de considerar overhead y ancho de banda.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Modelo de cutoff

**Situación.** Se estima cuándo el trabajo por tarea supera un overhead de creación medido.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
overhead_us = 3.2
cost_per_item_us = 0.08
candidates = (8, 16, 32, 64, 128, 256)
for items in candidates:
    useful = items * cost_per_item_us
    ratio = useful / overhead_us
    print(f"items={items:3} trabajo={useful:5.2f}us trabajo/overhead={ratio:4.1f}")
cutoff = next(items for items in candidates if items * cost_per_item_us >= 5 * overhead_us)
assert cutoff == 256
print("cutoff inicial:", cutoff)


### Lectura razonada

El factor cinco es una hipótesis de partida; el cutoff final se obtiene en el hardware objetivo.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Cómo detectarías experimentalmente que el cutoff genera tareas demasiado pequeñas?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Dibujar dependencias de mergesort o stencil.
2. Medir número de tareas y tiempo para varios cutoffs.
3. Comparar con una versión `parallel for` cuando la estructura lo permita.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Crear una región paralela por llamada recursiva.
- Omitir `single` al generar el DAG.
- Medir solo un tamaño y declarar un cutoff universal.


## Criterios de aceptación

- DAG sin carreras ni dependencias faltantes.
- Cutoff justificado con curva.
- Resultado comparado con versión serial.


## Síntesis

- La pregunta que debes poder responder es: **¿Cuándo un DAG de tareas expone paralelismo suficiente para compensar el costo de creación y sincronización?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Planeación OpenMP](../../../docs/PLANEACION_CURSO.md)
- [Ejemplos OpenMP](../../../openmp/)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 03](README.md)
